# Token Classification (NER)
```
0 = O
1 = B-PER
2 = I-PER
3 = B-ORG
4 = B-LOC


In [11]:
## ✅ 1. IMPORTS (same style)

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import numpy as np
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


In [12]:
## ✅ 2. DATASET (same structure, just renamed labels)

train_data = {
    "tokens": [
        ["John", "works", "at", "Google"],
        ["Alice", "lives", "in", "Paris"],
        ["Bob", "joined", "Microsoft"],
        ["Emma", "is", "from", "London"]
    ],
    "labels": [   # 👈 renamed from ner_tags
        [1, 0, 0, 3],
        [1, 0, 0, 4],
        [1, 0, 3],
        [1, 0, 0, 4]
    ]
}

test_data = {
    "tokens": [
        ["David", "works", "at", "Amazon"]
    ],
    "labels": [
        [1, 0, 0, 3]
    ]
}

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "test": Dataset.from_dict(test_data)
})


In [13]:
## ✅ 3. LABEL MAPPING (IMPORTANT FOR STUDENTS)

label_list = ["O", "B-PER", "I-PER", "B-ORG", "B-LOC"]

id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

NUM_LABELS = len(label_list)
MODEL_NAME = "bert-base-uncased"


In [15]:
## ✅ 4. TOKENIZER

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
## 🔥 5. PREPROCESS (MOST IMPORTANT PART)
### 👉 This is where NER differs from classification

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # ignore special tokens
            else:
                label_ids.append(label[word_id])

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

dataset = dataset.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [16]:
## ✅ 6. MODEL

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [17]:
## ✅ 7. TRAINING SETUP (same style as before)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True
)


In [18]:
## 🔥 8. METRICS (TOKEN-LEVEL)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)

    true_labels = []
    true_preds = []

    for pred, lab in zip(preds, labels):
        for p, l in zip(pred, lab):
            if l != -100:
                true_labels.append(l)
                true_preds.append(p)

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_preds, average="micro"
    )
    acc = accuracy_score(true_labels, true_preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }


In [19]:
## ✅ 9. DATA COLLATOR (NEW FOR NER)

data_collator = DataCollatorForTokenClassification(tokenizer)


In [21]:
## ✅ 10. TRAINER

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    # tokenizer=tokenizer, Not needed
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [22]:
## ✅ 11. TRAIN

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.634307,1.450871,0.500000,0.500000,0.500000,0.500000
2,1.518993,1.336363,0.500000,0.500000,0.500000,0.500000
3,1.318967,1.277308,0.500000,0.500000,0.500000,0.500000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3, training_loss=1.490755597750346, metrics={'train_runtime': 47.1326, 'train_samples_per_second': 0.255, 'train_steps_per_second': 0.064, 'total_flos': 3135646126080.0, 'train_loss': 1.490755597750346, 'epoch': 3.0})

In [23]:
## ✅ 12. SAVE

trainer.save_model("my_ner_model")
tokenizer.save_pretrained("my_ner_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my_ner_model/tokenizer_config.json', 'my_ner_model/tokenizer.json')

In [26]:
# 🔥 13. INFERENCE (SUPER IMPORTANT FOR DEMO)

from transformers import pipeline

ner_pipeline = pipeline(
    "token-classification",
    model="my_ner_model",
    tokenizer="my_ner_model",
    aggregation_strategy="simple",

)

# aggregation_strategy="simple"
# Merges tokens into full entities
# Filters out "O" (non-entity) tokens


text = "John works at Google in Paris"

results = ner_pipeline(text)

for r in results:
    print(r)

# “Pipeline hides ‘O’ labels because they are not useful in real applications.”
# So you may see only those tokens that are NOT O

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'entity_group': 'LOC', 'score': np.float32(0.26548156), 'word': 'google', 'start': 14, 'end': 20}


In [28]:
# If you want to see all the tokens:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to device
model.to(device)

text = "John works at Google in Paris"

# Tokenize
inputs = tokenizer(text, return_tensors="pt")

# 🔥 MOVE INPUTS TO SAME DEVICE
inputs = {k: v.to(device) for k, v in inputs.items()}

# Inference
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
predictions = logits.argmax(dim=2)

# Move back to CPU for printing
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].cpu())
pred_labels = [model.config.id2label[p.item()] for p in predictions[0].cpu()]

for token, label in zip(tokens, pred_labels):
    if token not in ["[CLS]", "[SEP]", "[PAD]"]:
        print(f"{token:12} → {label}")


john         → O
works        → O
at           → O
google       → B-LOC
in           → O
paris        → O



# Explain This to Students (Golden Line)

👉 Classification:

```
One sentence → One label
```

👉 NER:

```
One sentence → Label for EACH word
```

---

# ⚠️ Common Mistake (Explain This Clearly)

If you skip **label alignment**, model will:

* learn garbage
* give random outputs

👉 This is the *#1 reason NER fails for beginners*

